In [2]:
# ============================================
# ADVANCED ETL PIPELINE 
# ============================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import os

spark = SparkSession.builder.appName("ETL_Pipeline").getOrCreate()


In [3]:
# ============================================
# READ GENERATED DATA (VERY IMPORTANT)
# ============================================

orders_df = spark.read.option("header", True).csv("data/bronze/orders.csv")
customers_df = spark.read.option("header", True).csv("data/bronze/customers.csv")
products_df = spark.read.option("header", True).csv("data/bronze/products.csv")

In [4]:
# ============================================
# BRONZE LAYER
# ============================================

os.makedirs("data/sample_data", exist_ok=True)

# Save SAMPLE data (GitHub use)
orders_df.limit(1000).toPandas().to_csv("data/sample_data/orders_sample.csv", index=False)
customers_df.limit(1000).toPandas().to_csv("data/sample_data/customers_sample.csv", index=False)
products_df.limit(200).toPandas().to_csv("data/sample_data/products_sample.csv", index=False)

print("Bronze Layer Completed")

Bronze Layer Completed


In [5]:
# ============================================
# SILVER LAYER
# ============================================

# Data Cleaning
orders = orders_df.dropDuplicates(["order_id"])
customers = customers_df.dropDuplicates(["customer_id"])
products = products_df.dropDuplicates(["product_id"])

orders = orders.fillna({"quantity": 0})
customers = customers.fillna({"country": "Unknown"})
products = products.fillna({"price": 0})

orders = orders.withColumn("quantity", col("quantity").cast("int"))
products = products.withColumn("price", col("price").cast("double"))

orders = orders.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

In [12]:
# Joins
joined_df = orders.join(customers, "customer_id", "left") \
                  .join(products, "product_id", "left")

In [13]:
# Derived Columns
joined_df = joined_df.withColumn("total_amount", col("quantity") * col("price")) \
                     .withColumn("order_year", year("order_date")) \
                     .withColumn("order_month", month("order_date"))

In [14]:
# Window Function
window_spec = Window.partitionBy("customer_id").orderBy(col("order_date").desc())

joined_df = joined_df.withColumn("rank", row_number().over(window_spec))

In [15]:
# Save Silver
os.makedirs("data/silver", exist_ok=True)
joined_df.toPandas().to_csv("data/silver/retail.csv", index=False)

print("Silver Layer Completed")

Silver Layer Completed


In [7]:
# ============================================
# GOLD LAYER
# ============================================

silver_df = spark.read.option("header", True).csv("data/silver/retail.csv")

In [8]:
# Fix types
silver_df = silver_df.withColumn("total_amount", col("total_amount").cast("double")) \
                     .withColumn("order_year", col("order_year").cast("int")) \
                     .withColumn("order_month", col("order_month").cast("int"))

In [9]:
# Aggregations
revenue_country = silver_df.groupBy("country") \
    .agg(sum("total_amount").alias("total_revenue"))

monthly_revenue = silver_df.groupBy("order_year", "order_month") \
    .agg(sum("total_amount").alias("monthly_revenue"))

top_customers = silver_df.groupBy("customer_id", "name") \
    .agg(sum("total_amount").alias("spending")) \
    .orderBy(desc("spending"))

top_categories = silver_df.groupBy("category") \
    .agg(sum("total_amount").alias("revenue")) \
    .orderBy(desc("revenue"))

In [10]:
# Save Gold
os.makedirs("data/gold", exist_ok=True)

revenue_country.toPandas().to_csv("data/gold/revenue_country.csv", index=False)
monthly_revenue.toPandas().to_csv("data/gold/monthly_revenue.csv", index=False)
top_customers.toPandas().to_csv("data/gold/top_customers.csv", index=False)
top_categories.toPandas().to_csv("data/gold/top_categories.csv", index=False)

print("Gold Layer Completed")

Gold Layer Completed


In [11]:
# ============================================
# DATA QUALITY CHECK
# ============================================

print("Null Check:")
silver_df.select([count(when(col(c).isNull(), c)).alias(c) for c in silver_df.columns]).show()

print("Top Customers:")
top_customers.show(10)

# ============================================
# END
# ============================================

Null Check:
+----------+-----------+--------+--------+----------+------+----+-----+-------+-----------+------------+--------+-----+------------+----------+-----------+----+
|product_id|customer_id|order_id|quantity|order_date|status|name|email|country|signup_date|product_name|category|price|total_amount|order_year|order_month|rank|
+----------+-----------+--------+--------+----------+------+----+-----+-------+-----------+------------+--------+-----+------------+----------+-----------+----+
|         0|          0|       0|       0|         0|     0|   0|    0|      0|          0|           0|       0|    0|           0|         0|          0|   0|
+----------+-----------+--------+--------+----------+------+----+-----+-------+-----------+------------+--------+-----+------------+----------+-----------+----+

Top Customers:
+-----------+---------------+------------------+
|customer_id|           name|          spending|
+-----------+---------------+------------------+
|      66566|  Allis